# IT9203 — Data Ingestion with Auto Loader

**Project:** Top-Rated Movies per Genre Tracker  
**Dataset:** MovieLens 25M (GroupLens)  
**Platform:** Azure Databricks  
**Notebook:** 03 — Data Ingestion (10 marks)
**Tool:** Auto Loader (Structured Streaming)

## 1. Architecture

Auto Loader incrementally ingests new CSV files as they arrive in the input directory, inferring schema and landing data as Delta/Parquet in DBFS.

```
+-----------------------+     detect new file     +------------------------+
|                       | ----------------------> |                        |
|  /FileStore/movielens |    (Auto Loader)       |  /FileStore/movielens  |
|  /input/              |                         |  /raw/                 |
|                       |    schema inference     |                        |
|  ratings.csv          |     + infer + merge     |  ratings_cleaned.delta |
|  movies.csv           |                         |                        |
+-----------------------+                         +------------------------+
```

**Why Auto Loader instead of Kafka?**
- Databricks-native streaming ingestion
- Schema inference and evolution out of the box
- Exactly-once guarantees without checkpoint management
- Files can be replayed and backfilled easily

## 2. Dataset Overview

**Source:** MovieLens 25M Dataset  
**Files:** `ratings.csv` (1M rows) + `movies.csv` (62K rows)  
**Format:** CSV

### ratings.csv
| Column | Type | Description |
|---|---|---|
| userId | int | Unique user identifier |
| movieId | int | Unique movie identifier |
| rating | double | 0.5-5.0 (0.5 increments) |
| timestamp | long | Unix epoch seconds |

### movies.csv
| Column | Type | Description |
|---|---|---|
| movieId | int | Unique movie identifier |
| title | string | Movie title (includes year) |
| genres | string | Pipe-delimited genre list |

## 3. Configure and Run Auto Loader

Auto Loader reads CSV files from the input directory. When new files appear, they are automatically ingested.

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType, StringType
from pyspark.sql.functions import input_file_name, current_timestamp

INPUT_PATH = "/FileStore/movielens/input/"
RAW_PATH = "/FileStore/movielens/raw/"
CHECKPOINT_PATH = "/FileStore/movielens/checkpoint/"

ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True)
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

print(f"Input path:      {INPUT_PATH}")
print(f"Output path:     {RAW_PATH}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")

Input path:      /FileStore/movielens/input/
Output path:     /FileStore/movielens/raw/
Checkpoint path: /FileStore/movielens/checkpoint/


## 4. Ingest Ratings with Auto Loader

Auto Loader detects `ratings.csv` in the input directory and streams it into DBFS as Parquet.

In [0]:
# Read ratings.csv using Auto Loader (cloudFiles format)
ratings_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/ratings_schema") \
    .option("header", "true") \
    .option("delimiter", ",") \
    .schema(ratings_schema) \
    .load(INPUT_PATH)

# Write to DBFS as Parquet with append mode
query = ratings_stream.writeStream \
    .format("parquet") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/ratings_checkpoint") \
    .option("path", RAW_PATH) \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .start()

query.awaitTermination()
print("Ratings ingestion complete.")

Ratings ingestion complete.


## 5. Load and Verify Raw Data

In [0]:
# Read raw Parquet and show stats
raw_ratings = spark.read.parquet(RAW_PATH)
total_records = raw_ratings.count()
unique_users = raw_ratings.select("userId").distinct().count()
unique_movies = raw_ratings.select("movieId").distinct().count()
avg_rating = raw_ratings.agg({"rating": "avg"}).collect()[0][0]

print(f"Total records ingested: {total_records:,}")
print(f"Unique users:           {unique_users:,}")
print(f"Unique movies:          {unique_movies:,}")
print(f"Average rating:         {avg_rating:.2f}")

display(raw_ratings.limit(5))

Total records ingested: 25,062,518
Unique users:           183,136
Unique movies:          59,048
Average rating:         3.53


userId,movieId,rating,timestamp
1,296,5.0,1147880044
1,306,3.5,1147868817
1,307,5.0,1147868828
1,665,5.0,1147878820
1,899,3.5,1147868510


## 6. Load Movies CSV

Movies data is loaded directly (small file, no streaming needed).

In [0]:
movies_df = spark.read \
    .option("header", "true") \
    .option("delimiter", ",") \
    .schema(movies_schema) \
    .csv(f"{INPUT_PATH}movies.csv")

print(f"Movies loaded: {movies_df.count():,}")
display(movies_df.limit(5))

Movies loaded: 62,423


movieId,title,genres
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy


## 7. Merge Ratings with Movies

Join on movieId to enrich each rating with movie title and genre list.

In [0]:
from pyspark.sql.functions import from_unixtime, col

# Merge ratings with movies
merged_df = raw_ratings \
    .join(movies_df, on="movieId", how="left") \
    .withColumn("review_date", from_unixtime(col("timestamp")).cast("date"))

merged_count = merged_df.count()
print(f"Merged records: {merged_count:,}")
display(merged_df.limit(5))

Merged records: 25,062,518


movieId,userId,rating,timestamp,title,genres,review_date
296,1,5.0,1147880044,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,2006-05-17
306,1,3.5,1147868817,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama,2006-05-17
307,1,5.0,1147868828,Three Colors: Blue (Trois couleurs: Bleu) (1993),Drama,2006-05-17
665,1,5.0,1147878820,Underground (1995),Comedy|Drama|War,2006-05-17
899,1,3.5,1147868510,Singin' in the Rain (1952),Comedy|Musical|Romance,2006-05-17


## 8. Data Statistics

In [0]:
print("=== DATASET STATISTICS ===")
print(f"Total records:       {merged_count:,}")
print(f"Unique users:        {merged_df.select('userId').distinct().count():,}")
print(f"Unique movies:       {merged_df.select('movieId').distinct().count():,}")
print(f"Average rating:      {merged_df.agg({'rating': 'avg'}).collect()[0][0]:.2f}")

print("\nRating distribution:")
display(merged_df.groupBy("rating").count().orderBy("rating"))

=== DATASET STATISTICS ===
Total records:       25,062,518
Unique users:        183,136
Unique movies:       59,048
Average rating:      3.53

Rating distribution:


rating,count
null,62423
0.5,393068
1.0,776815
1.5,399490
2.0,1640868
2.5,1262797
3.0,4896928
3.5,3177318
4.0,6639798
4.5,2200539


## 9. Write Merged Dataset to DBFS

In [0]:
# Save merged dataset as Parquet for downstream notebooks
merged_df.write \
    .mode("overwrite") \
    .parquet("/FileStore/movielens/raw/merged_ratings/")

print("Merged dataset saved to /FileStore/movielens/raw/merged_ratings/")
display(dbutils.fs.ls("/FileStore/movielens/raw/merged_ratings/"))

Merged dataset saved to /FileStore/movielens/raw/merged_ratings/


path,name,size,modificationTime
dbfs:/FileStore/movielens/raw/merged_ratings/_SUCCESS,_SUCCESS,0,1779547011000
dbfs:/FileStore/movielens/raw/merged_ratings/_committed_5646748408922524642,_committed_5646748408922524642,824,1779547011000
dbfs:/FileStore/movielens/raw/merged_ratings/_started_5646748408922524642,_started_5646748408922524642,0,1779547006000
dbfs:/FileStore/movielens/raw/merged_ratings/part-00000-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-268-1.c000.snappy.parquet,part-00000-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-268-1.c000.snappy.parquet,34849953,1779547010000
dbfs:/FileStore/movielens/raw/merged_ratings/part-00001-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-269-1.c000.snappy.parquet,part-00001-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-269-1.c000.snappy.parquet,55329462,1779547010000
dbfs:/FileStore/movielens/raw/merged_ratings/part-00002-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-270-1.c000.snappy.parquet,part-00002-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-270-1.c000.snappy.parquet,37294310,1779547010000
dbfs:/FileStore/movielens/raw/merged_ratings/part-00003-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-264-1.c000.snappy.parquet,part-00003-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-264-1.c000.snappy.parquet,34384024,1779547010000
dbfs:/FileStore/movielens/raw/merged_ratings/part-00004-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-265-1.c000.snappy.parquet,part-00004-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-265-1.c000.snappy.parquet,34052624,1779547010000
dbfs:/FileStore/movielens/raw/merged_ratings/part-00005-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-266-1.c000.snappy.parquet,part-00005-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-266-1.c000.snappy.parquet,33147759,1779547010000
dbfs:/FileStore/movielens/raw/merged_ratings/part-00006-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-267-1.c000.snappy.parquet,part-00006-tid-5646748408922524642-0361348e-b401-44e1-b3a6-dfbd24c9624c-267-1.c000.snappy.parquet,33060548,1779547010000


## 10. Ingestion Summary

| Metric | Value |
|---|---|
| Tool | Auto Loader (Structured Streaming) |
| Source | `/FileStore/movielens/input/` |
| Destination | `/FileStore/movielens/raw/` + `merged_ratings/` |
| Records ingested | 25,062,518 |
| Format | Parquet (Snappy compressed) |
| Trigger | `availableNow` (batch-on-file-arrival) |

All data is now in DBFS. Proceed to Notebook 04 — Data Processing.